In [37]:
load_dotenv()

True

In [22]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import operator
import os


In [15]:
class EvalSchema(BaseModel):
    feedback:str=Field(description="detailed feedback for the essay")
    score:int=Field(description="score out of 10",ge=0,le=10)

In [16]:
class UState(TypedDict):
    essay_text:str
    cot_feedback:str
    doa_feedback:str
    lan_feedback:str
    final_feedback:str
    individual_score:Annotated[list[int],operator.add]
    final_score:float

In [ ]:
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",api_key=os.getenv("GEMINI_API_KEY"))
newmodel=model.with_structured_output(EvalSchema)

In [39]:
def cot(state:UState):
    essay=state["essay_text"]
    eval=newmodel.invoke(f"evaluate this essay{essay} on the basis of clarity of thought , and score it out of 10")
    return {"cot_feedback":eval.feedback,"individual_score":[eval.score]}


def evaldoa(state:UState):
    essay=state["essay_text"]
    eval=newmodel.invoke(f"evaluate this essay{essay} on the basis of depth of the analysis , and score it out of 10")
    return {"doa_feedback":eval.feedback,"individual_score":[eval.score]}

def evallan(state:UState):
    essay=state["essay_text"]
    eval=newmodel.invoke(f"evaluate this essay{essay} on the basis of language , and score it out of 10")
    return {"lan_feedback":eval.feedback,"individual_score":[eval.score]}

def final(state:UState):
    essay=state["essay_text"]
    eval=model.invoke(f"evaluate this essay{essay} on the basis of  following feedbacks create a summarised feedback clarity of thought feedback{state['cot_feedback']}  , language feedback{state['lan_feedback']},depth of analysis feedback {state['doa_feedback']}")
    score=sum(state["individual_score"])/3
    return {"final_feedback":eval.content,"final_score":score}

In [40]:
graph=StateGraph(UState)
graph.add_node("evalcot",cot)
graph.add_node("evaldoa",evaldoa)
graph.add_node("evallan",evallan)
graph.add_node("finalfeedback",final)
graph.add_edge(START,"evalcot")
graph.add_edge(START,"evaldoa")
graph.add_edge(START,"evallan")
graph.add_edge("evalcot","finalfeedback")
graph.add_edge("evaldoa","finalfeedback")
graph.add_edge("evallan","finalfeedback")
graph.add_edge("finalfeedback",END)
workflow=graph.compile()



In [41]:

essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""
istate={"essay_text":essay2}
fstate=workflow.invoke(istate)
fstate

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 53.921832354s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT